# M12 on Google Colab: the adaptation-policy agent on E4 (`e4_added`, D11), full pool

The Colab version of `m12_kaggle.ipynb`, for when the Kaggle GPU quota is used up. Same scripts and settings.

**Before running:**
1. *Runtime → Change runtime type →* **T4 GPU**.
2. Upload the five files of the M1 `processed/` folder (the same files as the Kaggle dataset) to a folder in your Google Drive, e.g. `MyDrive/ids/processed/`. The notebook finds it through `data_manifest.json`.
3. Hugging Face: accept the Gemma 2 licence for `google/gemma-2-2b-it` on your account, then add your token in Colab's *Secrets* (key icon) as **`HF_TOKEN`**, with notebook access on.

**Run:** *Runtime → Run all*, then allow Drive access. Keep the tab open: free Colab disconnects idle sessions.

**Output:** `MyDrive/ids_outputs/m12_full_seed43_outputs.zip`, written after every arm. **After a disconnect, just run all again:** it restores that zip and skips the arms already done.

Differences from Kaggle, recorded per run in `environment.json`: 2 vCPUs (the fits still use 4 threads, as M9 did, so they compute the same thing, only slower), about 12.7 GB RAM, and Colab's own Python, PyTorch and Transformers. XGBoost, LightGBM and scikit-learn are pinned to the versions of the Kaggle runs. The none arm's comparison with M9 shows whether anything else changed.

In [ ]:
SEED = 43
TAG = "" if SEED == 42 else f"seed{SEED}"   # seed 42 is the main report, as in M7
TAG_ARG = f"--tag {TAG}" if TAG else ""
import glob, os, shutil, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/ids_outputs"
os.makedirs(OUT, exist_ok=True)
REPO = "/content/IDS-code"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Indrajit147/IDS-code.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
# The tree libraries pinned to the Kaggle runs' versions; --no-deps keeps Colab's stack otherwise intact.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xgboost==3.4.1", "lightgbm==4.6.0", "scikit-learn==1.6.1", "bitsandbytes"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO, "--no-deps"], check=True)
os.chdir(REPO)
sys.path.insert(0, f"{REPO}/src")
os.environ["IDS_REPO_ROOT"] = REPO
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
assert os.path.exists("configs/env/colab.yaml"), "This clone predates the Colab profile. Runtime -> Disconnect and delete runtime, then run all again."
# The data: copy the Drive folder holding data_manifest.json to local disk (Parquet through Drive is slow).
DATA = "/content/data/processed"
if not os.path.exists(f"{DATA}/data_manifest.json"):
    hits = glob.glob("/content/drive/MyDrive/**/data_manifest.json", recursive=True)
    assert hits, "No data_manifest.json in your Drive: upload the M1 processed/ files (see the top of this notebook)."
    shutil.copytree(os.path.dirname(hits[0]), DATA, dirs_exist_ok=True)
print("data:", sorted(os.listdir(DATA)))

In [ ]:
# The model from Hugging Face (needs the HF_TOKEN secret and the accepted Gemma 2 licence)
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
MODEL = "google/gemma-2-2b-it"
LLM_SET = f"policy.backend=hf llm.model_id={MODEL}"

In [ ]:
# GPU smoke test: 4 policy decisions on made-up evidence. Stops here on failure.
import json, time, torch
assert torch.cuda.is_available(), "No GPU: set Accelerator to GPU T4"
from ids.llm.policy import decide
from ids.llm.verifier import HFBackend
be = HFBackend(MODEL, "4bit", "float16", 256)
EV = {"adwin_fired": True, "fired_at_round_share": 0.3, "rows_new": 9000, "error_rate_new_rows": 0.21, "error_rate_validation": 0.05,
      "new_classes": ["recon"], "class_share_new_rows": {"dos": 0.6, "recon": 0.4}, "class_share_buffer": {"benign": 0.3, "dos": 0.7},
      "per_class_f1_validation": {"benign": 0.99, "dos": 0.97}, "feature_shift_share": 0.3, "most_shifted_features": ["Rate", "syn_count", "AVG", "Std", "UDP"],
      "shap_drift": 0.42, "top_shap_features_new_rows": ["Rate", "Header_Length", "syn_count", "AVG", "Number"], "previous_policy": None}
tic = time.perf_counter()
recs = [decide(EV, be, 2) for _ in range(4)]
sec = time.perf_counter() - tic
print(json.dumps(recs[0]["policy"], indent=1))
print("valid:", sum(r["valid"] for r in recs), "of 4; first try:", sum(r["first_try_valid"] for r in recs), f"; {sec / 4:.1f} s per decision")
del be; torch.cuda.empty_cache()
assert sum(r["valid"] for r in recs) >= 3, "The model's replies mostly fail the contract: stop and bring the output back"

In [ ]:
# Rebuild the M2 partition for the full pool (deterministic)
!python scripts/m2_build_partitions.py --env colab --pools full
!git checkout -- 'reports/m2/full/*.png'

In [ ]:
import zipfile
REP = os.path.join("reports/m12/full", TAG)
ZIP = f"{OUT}/m12_full_seed{SEED}_outputs.zip"
def pack():
    stage = "/content/m12_stage"
    shutil.rmtree(stage, ignore_errors=True)
    os.makedirs(stage)
    if os.path.isdir(REP):
        shutil.copytree(REP, f"{stage}/{REP}")
    for r in sorted(glob.glob("/content/results/runs/*_m12_policy_full")):
        shutil.copytree(r, f"{stage}/results/runs/{os.path.basename(r)}")
    shutil.make_archive(ZIP[:-4], "zip", stage)
    shutil.rmtree(stage)
    print("zip written to Drive:", sorted(os.listdir(REP)) if os.path.isdir(REP) else [])
# Resume: restore an earlier session's outputs from Drive (reports and run folders)
if os.path.exists(ZIP):
    with zipfile.ZipFile(ZIP) as z:
        for n in z.namelist():
            if n.startswith(REP + "/"):
                z.extract(n, REPO)
            elif n.startswith("results/runs/"):
                z.extract(n, "/content")
    print("restored:", sorted(os.listdir(REP)) if os.path.isdir(REP) else [])

In [ ]:
# the three arms, each in its own process; arms already done (restored from Drive) are skipped
for arm in ["none", "heuristic", "llm"]:
    if os.path.exists(f"{REP}/arm_{arm}.json"):
        print("=====", arm, "already done")
        continue
    print("=====", arm, flush=True)
    !python -X faulthandler scripts/m12_policy.py --env colab --pool full --set resources.n_jobs=4 seed={SEED} {LLM_SET} {TAG_ARG} --only {arm}; echo "exit code: $?"
    pack()
!python scripts/m12_policy.py --env colab --pool full --set resources.n_jobs=4 seed={SEED} {LLM_SET} {TAG_ARG} --report; echo "exit code: $?"
pack()
if os.path.exists(f"{REP}/POLICY.md"):
    print(open(f"{REP}/POLICY.md", encoding="utf-8").read())